# Dados aeroportuários auditáveis — cinco experimentos com a pista do SBSJ

**Pergunta.** É possível tornar dados aeroportuários geolocalizados auditáveis — cada valor atribuível, guardado, recalculável a partir das premissas e com adulteração evidente? E onde estão os limites dessa garantia?

Os experimentos usam a biblioteca [`aeroledger`](https://github.com/heyfalheiro1537/aeroledger) como qualquer usuário usaria: cada passo é uma chamada a ela.

| | Pergunta |
|---|---|
| **E1** | Qual é o problema nas publicações de hoje? |
| **E2** | O número se refaz a partir das premissas? |
| **E3** | Como estava em *t*, quem afirmou e por quê? |
| **E4** | Alguém consegue adulterar o banco sem ser visto? |
| **E5** | Dois produtos do mesmo banco podem divergir? A prova é verificável por terceiros? |

**Como ler.** Cada experimento mostra os resultados. As verificações — o que se esperava, com que margem e por quê — ficam juntas no fim, em células separadas, para que os resultados apareçam mesmo que alguma verificação falhe.

Fontes: AIP Brasil, AD 2.12 e AD 2.13 (eAIP 20-2026); Carta de Aeródromo AIRAC 2604A1; Carta de Obstáculos AIRAC 2213A1. Uso em pesquisa: nada aqui substitui a publicação oficial.

In [ ]:
import json
import tempfile
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import aeroledger as al
from aeroledger import adc, derivacoes
from aeroledger.referencial import Resolvedor

import apoio
import ataques
import fontes as F
import geodesia as gabarito                  # outro código, sem a biblioteca
import verificador_independente as vi        # idem
from apoio import seed as s

def tabela(cabecalho, linhas):
    display(Markdown("| " + " | ".join(cabecalho) + " |\n|" + "---|" * len(cabecalho) + "\n" +
                     "\n".join("| " + " | ".join(str(c) for c in l) + " |" for l in linhas)))

conn = apoio.banco_novo()
print("aeroledger", al.__version__, "· banco", apoio.BANCO, "recriado vazio")

## E1 — Qual é o problema nas publicações de hoje?

Primeiro, a pista registrada **como o AIXM faria**: cada número publicado vira um campo guardado. Um registro com a Carta de Obstáculos de 2022 e uma alteração com a Carta de Aeródromo de 2026, ambos afirmados pelo DECEA. Fica numa execução à parte (`aixm`), porque é o modelo que o trabalho critica.

In [ ]:
RUN_F = al.novo_run(conn, F.AIRAC_2213, "aixm")
e2213, _ = al.escrever(conn, RUN_F, "SBSJ-RWY-AIXM", "runway_forma", "BASELINE", corpo=F.FORMA_AIRAC_2213,
                       asserted_by="DECEA", reason="AOC AIRAC AMDT 2213A1")
apoio.avancar_ate(conn, RUN_F, F.AIRAC_2604)
antes, depois = al.ledger.achatar(F.FORMA_AIRAC_2213), al.ledger.achatar(F.FORMA_AIRAC_2604)
e2604, _ = al.escrever(conn, RUN_F, "SBSJ-RWY-AIXM", "runway_forma", "DELTA",
                       alteracoes={p: v for p, v in depois.items() if antes[p] != v},
                       asserted_by="DECEA", reason="ADC AIRAC AMDT 2604A1")

E1_DIFF = al.diff(conn, RUN_F, e2213, e2604)
E1_ENVELOPE = al.envelope(conn, e2604)
tabela(["campo", "2022 (AOC)", "2026 (ADC)"], [(f"`{p}`", a, b) for _, p, a, b in E1_DIFF])
print(f"{len(E1_DIFF)} campos mudaram · causa registrada: {E1_ENVELOPE['caused_by']} · motivo: {E1_ENVELOPE['reason']!r}")

Uma única mudança física — a pista passou de 2676 para 2675 m — aparece como **sete números editados separadamente**, porque TORA, TODA, ASDA e LDA estão guardados em vez de calculados. E o registro diz quem publicou, não por que o número mudou.

## E2 — O número se refaz a partir das premissas?

Agora o SBSJ no modelo proposto (execução `sbsj`). Entram só as **premissas**: as coordenadas das duas cabeceiras do AD 2.12, a largura da pista e a zona de parada. Toda posição se apoia nas cabeceiras — a pista vai "da THR16 até a THR34" —, e o número 2675 não é escrito em lugar nenhum.

| Papel | Dado | Fonte |
|---|---|---|
| premissa | coordenadas e elevações das cabeceiras | AD 2.12 |
| premissa | largura da pista; dimensões da zona de parada | AD 2.12 |
| norma | variação magnética; regra das distâncias declaradas | ADC; Anexo 14 |
| derivado | comprimento, azimutes, rumos, TORA/TODA/ASDA/LDA | calculado, nunca escrito |

In [ ]:
RUN = al.novo_run(conn, F.EAIP_20_2026, "sbsj")
REF1 = al.publicar_referencial(conn, apoio.referencial())
CORPO1 = al.carregar_referencial(conn, REF1)

al.escrever(conn, RUN, "SBSJ", "airport", "BASELINE", asserted_by="DECEA", reason="eAIP 20-2026 (AD 2); ADC AIRAC 2604A1",
            corpo={"adc": {"icao": "SBSJ"}, "referencial": {"ref_id": REF1},
                   "norma": {"variacao_magnetica_graus": F.VARIACAO_LESTE}})
al.escrever(conn, RUN, "SBSJ-RWY", "runway", "BASELINE", asserted_by="DECEA", reason="eAIP 20-2026, AD 2.12",
            corpo={"adc": {"designacao": "16/34", "direcoes": {"inicio": "16", "fim": "34"}, "superficie": F.SUPERFICIE},
                   "geometria": {"primitivos": {"pista": {"tipo": "retangulo", "inicio": s("THR16"), "fim": s("THR34"),
                                                          "largura": F.LARGURA_RWY}}}})
for d, ref in (("16", "THR16"), ("34", "THR34")):
    al.escrever(conn, RUN, f"SBSJ-THR{d}", "runway_direction", "BASELINE", asserted_by="DECEA", reason="eAIP 20-2026, AD 2.12",
                corpo={"adc": {"designador": d, "pista": "SBSJ-RWY"},
                       "geometria": {"primitivos": {"thr": {"tipo": "ponto", "seed": s(ref)}}}})
al.escrever(conn, RUN, "SBSJ-SWY", "protect_area", "BASELINE", asserted_by="DECEA", reason="eAIP 20-2026, AD 2.12",
            corpo={"adc": {"papel": "swy", "aplica_a": {"pista": "SBSJ-RWY", "direcao": "16"}},
                   "geometria": {"primitivos": {"swy": {"tipo": "retangulo", "inicio": s("THR34"),
                                                        "fim": s("THR34", -F.SWY["comprimento"]),
                                                        "largura": F.SWY["largura"]}}}})
T_A = al.agora(conn, RUN)
print("premissas escritas em", T_A)

O desenho abaixo **não está no banco**: é gerado na leitura, a partir das cabeceiras e das posições relativas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for el in adc.projetar(conn, RUN)["elementos"]:
    if el["forma"] is not None:
        x, y = el["forma"].exterior.xy
        ax.fill(x, y, alpha=0.6, label=el["feature_id"])
    for p in el["nos"]:
        ax.plot(p.x, p.y, "k^")
ax.set_aspect("equal"); ax.set_xlabel("leste (m)"); ax.set_ylabel("norte (m)")
ax.set_title("Pista e zona de parada geradas das premissas (plano local na THR16)")
ax.legend(loc="lower left"); plt.show()

### Os valores calculados e os publicados

In [ ]:
L1 = derivacoes.comprimento(CORPO1, "RWY_16_34")
AZ1 = derivacoes.azimutes(CORPO1, "RWY_16_34")
RUMO = {d: derivacoes.rumo_magnetico(AZ1[f"THR{d}"], F.VARIACAO_LESTE) for d in ("16", "34")}
DD1 = al.distancias_declaradas(conn, RUN)

PUBLICADO = {"comprimento": 2675, "azimute 16": 134.93, "azimute 34": 314.92, "rumo 16": 157, "rumo 34": 337,
             "TORA 16": 2675, "TODA 16": 2675, "ASDA 16": 2958, "LDA 16": 2675,
             "TORA 34": 2675, "TODA 34": 2675, "ASDA 34": 2675, "LDA 34": 2675}
CALCULADO = {"comprimento": L1, "azimute 16": AZ1["THR16"], "azimute 34": AZ1["THR34"],
             "rumo 16": RUMO["16"], "rumo 34": RUMO["34"],
             **{f"{k} {d}": DD1[d][k] for d in ("16", "34") for k in ("TORA", "TODA", "ASDA", "LDA")}}
tabela(["grandeza", "calculado", "publicado", "diferença"],
       [(g, f"{CALCULADO[g]:.3f}", PUBLICADO[g], f"{CALCULADO[g] - PUBLICADO[g]:+.3f}") for g in PUBLICADO])

E2_DERIVADOS_NO_BANCO = apoio.contar(conn, r'''SELECT count(*) FROM event_field f JOIN event e USING (event_id)
    WHERE e.run_id = %s AND regexp_replace(f.path, '^.*\.', '') IN
          ('tora','toda','asda','lda','azimute','rumo','comprimento')''', RUN)
print("campos de valor derivado guardados no banco:", E2_DERIVADOS_NO_BANCO)

### A cabeceira 34 é remedida

Cenário simulado: um novo levantamento desloca a THR34 em cerca de 1 m. O levantamento entra como evento próprio, e a troca de referencial aponta para ele como causa. Nenhuma posição de pista ou de zona de parada é reescrita.

In [ ]:
al.avancar(conn, RUN, dias=30)
MARCO = apoio.ultimo_event_id(conn, RUN)
LEV, _ = al.escrever(conn, RUN, "SBSJ-LEV-THR34", "survey", "BASELINE",
                     asserted_by="Heyder (cenário simulado)", reason="remedição da cabeceira 34",
                     corpo={"levantamento": {"ponto": "THR34", "lat_publicada": F.THR34_REMEDIDA["lat"],
                                             "lon_publicada": F.THR34_REMEDIDA["lon"]}})
REF2 = al.publicar_referencial(conn, apoio.referencial(F.THR34_REMEDIDA, "levantamento SBSJ-LEV-THR34"))
CORPO2 = al.carregar_referencial(conn, REF2)
TROCA, _ = al.escrever(conn, RUN, "SBSJ", "airport", "DELTA", alteracoes={"referencial.ref_id": REF2},
                       asserted_by="DECEA", reason="troca de referencial: THR34 remedida", caused_by=LEV)
E2_ESCRITOS = apoio.escritos_depois_de(conn, RUN, MARCO)
tabela(["feature", "tipo", "evento", "campos escritos"],
       [(f, t, et, ", ".join(p)) for _, f, t, et, p in E2_ESCRITOS])

L2 = derivacoes.comprimento(CORPO2, "RWY_16_34")
AZ2 = derivacoes.azimutes(CORPO2, "RWY_16_34")
print(f"comprimento: {L1:.3f} m → {L2:.3f} m, sem nenhum número digitado")

**Conferência com o gabarito.** Os cantos da pista e da zona de parada, calculados pela biblioteca com o referencial antigo e com o novo, comparados com o mesmo cálculo feito por outro código (fórmulas de Vincenty, `geodesia.py`).

In [ ]:
E2_VERTICES = []
for nome, corpo in (("antes", CORPO1), ("depois", CORPO2)):
    res = Resolvedor(corpo)
    for ref, s_, o_ in F.VERTICES:
        outro = "THR34" if ref == "THR16" else "THR16"
        lat, lon, _ = res.seed(s(ref, s_, o_))
        glat, glon = gabarito.resolver(apoio.latlon(corpo, ref), apoio.latlon(corpo, outro), s_, o_)
        E2_VERTICES.append((nome, f"{ref} s={s_} o={o_}", gabarito.inversa(lat, lon, glat, glon)[0]))
tabela(["referencial", "canto", "distância biblioteca × gabarito (m)"], [(a, b, f"{c:.2e}") for a, b, c in E2_VERTICES])

gL, gaz16, gaz34 = gabarito.inversa(*apoio.latlon(CORPO2, "THR16"), *apoio.latlon(CORPO2, "THR34"))
E2_DL = abs(L2 - gL)
E2_DAZ = max(abs(AZ2["THR16"] - gaz16), abs(AZ2["THR34"] - gaz34))
print(f"comprimento novo: biblioteca {L2:.4f} m · gabarito {gL:.4f} m · Δ = {E2_DL:.1e} m; maior Δ de azimute {E2_DAZ:.1e}°")

## E3 — Como estava em *t*, quem afirmou e por quê?

As perguntas de um auditor, respondidas só lendo o banco:

- **estado em *t***: antes da remedição vale o referencial antigo e o comprimento antigo; agora, os novos;
- **ciclo de vida**: uma interdição temporária de 300 m a partir da THR34 entra num dia e sai três dias depois, sem nenhum tipo especial de "temporário";
- **quem e por quê**: o envelope da troca de referencial dá autor, motivo e causa;
- **o número está certo**: as distâncias recalculadas para o passado são as que valiam no passado.

In [ ]:
al.avancar(conn, RUN, dias=10)
T4 = al.agora(conn, RUN)
B4, _ = al.escrever(conn, RUN, "SBSJ-RES-0001", "restriction", "BASELINE", owner_ref="SBSJ-RWY",
                    asserted_by="operador SBSJ", reason="NOTAM (simulado)",
                    corpo={"adc": {"texto": "RWY 16/34: 300 m a partir da THR 34 interditados para obra",
                                   "trecho": {"de": s("THR34"), "ate": s("THR34", 300)}}})
al.avancar(conn, RUN, dias=3)
T5 = al.agora(conn, RUN)
V5, _ = al.escrever(conn, RUN, "SBSJ-RES-0001", "restriction", "VOID", owner_ref="SBSJ-RWY",
                    asserted_by="operador SBSJ", reason="obra concluída")
al.avancar(conn, RUN, dias=1)
AGORA = al.agora(conn, RUN)

ref_a, corpo_a = al.referencial_vigente(conn, RUN, al.relogio.limite(T_A))
ref_b, corpo_b = al.referencial_vigente(conn, RUN, al.relogio.limite(AGORA))
E3_ESTADO = {"t_a": (ref_a, derivacoes.comprimento(corpo_a, "RWY_16_34")),
             "agora": (ref_b, derivacoes.comprimento(corpo_b, "RWY_16_34"))}
tabela(["instante", "referencial vigente", "comprimento derivado (m)"],
       [(k, f"`{r[:12]}…`", f"{L:.3f}") for k, (r, L) in E3_ESTADO.items()])

def interditada(t):
    return any(f == "SBSJ-RES-0001" for f, _ in al.features(conn, RUN, t, tipos=["restriction"]))

E3_DENTRO = [T4, T4 + timedelta(days=1.5), apoio.um_segundo_antes(T5)]
E3_FORA = [apoio.um_segundo_antes(T4), T5, AGORA]
tabela(["instante", "interditada?"], [(f"{t:%d/%m %H:%M:%S}", interditada(t)) for t in sorted(E3_DENTRO + E3_FORA)])

E3_CICLOS = al.ciclos(conn, RUN, "SBSJ-RES-0001")
E3_ENV_TROCA, E3_ENV_LEV = al.envelope(conn, TROCA), al.envelope(conn, LEV)
print("ciclo de vida da interdição:", [(tipo, ciclo, f"{al.instante(e):%d/%m}") for e, tipo, ciclo in E3_CICLOS])
print(f"troca de referencial: afirmada por {E3_ENV_TROCA['asserted_by']}, motivo '{E3_ENV_TROCA['reason']}', "
      f"causa {E3_ENV_LEV['feature_id']} (afirmada por {E3_ENV_LEV['asserted_by']})")

E3_DD_TA = al.distancias_declaradas(conn, RUN, T_A)
E3_TRILHA = al.diff(conn, RUN, T_A, AGORA)
display(Markdown("**Trilha de auditoria entre t_a e agora**"))
tabela(["feature", "campo", "antes", "depois"], [(f, f"`{p}`", str(a)[:20], str(b)[:20]) for f, p, a, b in E3_TRILHA])

## E4 — Alguém consegue adulterar o banco sem ser visto?

O PANS-AIM (2.2.1) recomenda garantir a integridade dos dados com criptografia, como funções de hash. Aqui o atacante tem acesso direto ao banco e não passa pela biblioteca. Cada ataque roda numa **cópia** do banco. Antes, anota-se fora do banco o hash do último evento: a **âncora**.

In [ ]:
E4_ANCORA = al.cabeca(conn, RUN)
E4_K, E4_K_SEGUINTE, _ = ataques.escolher_k(conn, RUN)
conn.close()                               # o PostgreSQL só copia um banco sem conexões abertas

E4 = {}
for ataque in ataques.ATAQUES:
    c2 = apoio.copia(apoio.BANCO, "aeroledger_ataque")
    ataques.atacar(c2, RUN, ataque, E4_K)
    E4[ataque] = ataques.detectar(c2, RUN, E4_ANCORA)
    c2.close(); apoio.apagar("aeroledger_ataque")

conn = apoio.reconectar()
print(f"âncora {E4_ANCORA[1][:16]}… · evento atacado k = {E4_K}")
tabela(["ataque", "verificar_cadeia aponta", "com a âncora"],
       [(ataques.ATAQUES[a], vc or "nada", "detecta" if ca else "não detecta") for a, (vc, ca) in E4.items()])

A última linha é o limite: quem controla o banco inteiro e recalcula todos os hashes passa pela verificação interna. Só a âncora, guardada fora do banco, o denuncia.

**Limites que este trabalho não testa.** O banco registra quem afirmou, mas não prova (não há assinatura digital por evento). E nenhum sistema torna uma medição verdadeira: se a fonte mede errado, o banco só atribui o erro a quem mediu — como as edições que divergem no E1.

## E5 — Dois produtos do mesmo banco podem divergir? A prova é verificável por terceiros?

O PANS-AIM (2.1.3.2) exige consistência entre os produtos de informação aeronáutica. No E1, os produtos reais divergem. Aqui, dois produtos saem do mesmo banco:

- **P1**, a tabela do AIP: comprimento e distâncias declaradas;
- **P2**, o conteúdo da carta de aeródromo (a lente ADC).

Como os dois saem da mesma derivação, a consistência vem por construção. A parte que não é óbvia vem depois: um **verificador independente** (`verificador_independente.py`), escrito só a partir da especificação e sem importar a biblioteca, confere a cadeia e o comprimento a partir de um JSON exportado.

In [ ]:
def p1(t):
    _, corpo = al.referencial_vigente(conn, RUN, al.relogio.limite(t))
    return {"comprimento": derivacoes.comprimento(corpo, "RWY_16_34"), "distancias": al.distancias_declaradas(conn, RUN, t)}

def p2(t):
    return {"distancias": adc.projetar(conn, RUN, t)["distancias"]}

E5 = {"P1 t_a": p1(T_A), "P2 t_a": p2(T_A), "P1 agora": p1(AGORA), "P2 agora": p2(AGORA)}
tabela(["produto", "ASDA 16 (m)", "LDA 34 (m)"],
       [(k, f"{v['distancias']['16']['ASDA']:.3f}", f"{v['distancias']['34']['LDA']:.3f}") for k, v in E5.items()])

EXPORT = Path(tempfile.mkdtemp()) / "sbsj.json"
apoio.exportar(conn, RUN, EXPORT)
dados = json.loads(EXPORT.read_text())
E5_VERIFICADOR = vi.verificar(dados)
E5_CABECA = list(al.cabeca(conn, RUN))
E5_L_VERIFICADOR = vi.comprimento(dados)
print(f"verificador independente: íntegra = {E5_VERIFICADOR['integra']}, "
      f"cabeça {E5_VERIFICADOR['cabeca'][1][:16]}… (biblioteca: {E5_CABECA[1][:16]}…)")
print(f"comprimento: verificador {E5_L_VERIFICADOR:.4f} m · biblioteca {E5['P1 agora']['comprimento']:.4f} m")

E5_ALVO = next(e for e in dados["eventos"] if e["feature_id"] == "SBSJ-RWY")
for campo in E5_ALVO["campos"]:
    if campo["path"].endswith("pista.largura"):
        campo["next"] += 1
E5_ADULTERADO = vi.verificar(dados)
print(f"largura da pista alterada no JSON → o verificador acusa o evento {E5_ADULTERADO['primeiro_divergente']}")

---

# Verificações

Cada célula abaixo confere um experimento com `assert`. O valor esperado e a margem estão na própria linha, com a origem no comentário. As margens nunca foram escolhidas: saem da precisão com que o AIP publica cada número ou da exatidão do gabarito.

Se uma verificação falhar, os resultados acima continuam visíveis, e a mensagem diz o valor observado.

In [ ]:
# E1 — o problema
# A pista, a SWY e a faixa mudaram entre as edições; em "modo forma" isso aparece como
# 12 campos: comprimento da pista, SWY, comprimento e largura da faixa, e as 8 distâncias declaradas.
assert len(E1_DIFF) == 12, f"{len(E1_DIFF)} campos"
assert {p.split(".")[0] for _, p, _, _ in E1_DIFF} == {"pista", "swy", "faixa", "distancias"}
# Nenhuma mudança traz a causa: só a edição que a publicou.
assert E1_ENVELOPE["caused_by"] is None
print("E1 ok")

In [ ]:
# E2 — os valores calculados batem com o AIP
# Margens pela precisão de publicação (PANS-AIM, App. 2):
#   comprimento e distâncias: 0,5 m (publicados ao metro) + 0,419 m (coordenadas das cabeceiras a 0,01")
#   ASDA 16: o anterior + 0,5 m da SWY, publicada ao metro
#   azimute: 0,005° (publicado a 0,01°) + 0,009° (efeito das coordenadas no rumo)
#   rumo magnético: 0,5° (rumo inteiro) + 0,5° (variação inteira) + 0,009°
MARGEM = {"comprimento": 0.919, "azimute": 0.014, "rumo": 1.009, "distancia": 0.919, "ASDA 16": 1.419}

def margem(g):
    if g == "ASDA 16":
        return MARGEM["ASDA 16"]
    return MARGEM[g.split()[0]] if g.split()[0] in MARGEM else MARGEM["distancia"]

for g, pub in PUBLICADO.items():
    dif = abs(CALCULADO[g] - pub)
    if g.startswith(("azimute", "rumo")):
        dif = min(dif, 360 - dif)
    assert dif <= margem(g), f"{g}: calculado {CALCULADO[g]:.3f}, publicado {pub}, |Δ| {dif:.3f} > {margem(g)}"

# Nenhum valor derivado foi guardado no banco.
assert E2_DERIVADOS_NO_BANCO == 0

# A remedição escreveu só o levantamento e a troca de referencial, com o levantamento como causa.
assert [(f, et) for _, f, _, et, _ in E2_ESCRITOS] == [("SBSJ-LEV-THR34", "BASELINE"), ("SBSJ", "DELTA")]
assert E2_ESCRITOS[1][4] == ["referencial.ref_id"]
assert al.envelope(conn, TROCA)["caused_by"] == LEV
# Nenhuma feature com posição (pista, cabeceiras, SWY) foi reescrita.
assert not any(t in ("runway", "runway_direction", "protect_area") for _, _, t, _, _ in E2_ESCRITOS)

# A biblioteca coincide com o gabarito (outro código) dentro da exatidão de Vincenty: 0,5 mm.
assert max(d for _, _, d in E2_VERTICES) <= 0.0005
assert E2_DL <= 0.0005
assert E2_DAZ <= 0.0005 / L2 * 180 / 3.141592653589793      # 0,5 mm na ponta da pista, em graus
print("E2 ok")

In [ ]:
# E3 — como estava em t, quem afirmou e por quê
# Antes da remedição vale o referencial antigo e o comprimento antigo; agora, os novos.
assert E3_ESTADO["t_a"] == (REF1, L1) and E3_ESTADO["agora"] == (REF2, L2)
# A interdição existe em [t4, t5) e não existe fora.
assert all(interditada(t) for t in E3_DENTRO) and not any(interditada(t) for t in E3_FORA)
# O ciclo de vida é exatamente: criada em t4, encerrada em t5.
assert E3_CICLOS == [(B4, "BASELINE", 1), (V5, "VOID", 1)]
assert al.instante(B4) == T4 and al.instante(V5) == T5
# O envelope responde quem, por quê e com qual causa.
assert E3_ENV_TROCA["asserted_by"] == "DECEA" and E3_ENV_TROCA["reason"] and E3_ENV_TROCA["caused_by"] == LEV
assert E3_ENV_LEV["asserted_by"]
# As distâncias recalculadas para t_a são, bit a bit, as que valiam em t_a.
assert E3_DD_TA == DD1
print("E3 ok")

In [ ]:
# E4 — adulteração
# Alterar, apagar ou inserir no meio: a verificação interna aponta o evento.
assert E4["alterar"] == ([E4_K], True)              # aponta o próprio k
assert E4["apagar"] == ([E4_K_SEGUINTE], True)      # o seguinte perde o elo com o anterior
assert E4["inserir"] == ([E4_K_SEGUINTE], True)     # idem: o seguinte não aponta para o inserido
# Reescrever tudo de k até o fim: a verificação interna não percebe; só a âncora externa.
assert E4["reescrever"] == ([], True)
print("E4 ok")

In [ ]:
# E5 — consistência entre produtos e verificação por terceiros
# Os dois produtos dão as mesmas distâncias, antes e depois da remedição...
assert E5["P1 t_a"]["distancias"] == E5["P2 t_a"]["distancias"]
assert E5["P1 agora"]["distancias"] == E5["P2 agora"]["distancias"]
# ...e mudam juntos.
assert E5["P1 t_a"]["distancias"] != E5["P1 agora"]["distancias"]
# O verificador independente confirma a cadeia e chega ao mesmo último hash.
assert E5_VERIFICADOR["integra"] and list(E5_VERIFICADOR["cabeca"]) == E5_CABECA
# E o mesmo comprimento, dentro da exatidão de Vincenty (0,5 mm).
assert abs(E5_L_VERIFICADOR - E5["P1 agora"]["comprimento"]) <= 0.0005
# Alterado um campo do arquivo exportado, ele aponta o evento alterado.
assert E5_ADULTERADO["primeiro_divergente"] == E5_ALVO["event_id"]
print("E5 ok")